In [ ]:
from Tools import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
import scipy.stats

# LaTex Formatierung in Plots
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Computer Modern"
})

pd.set_option("display.precision", 10)
pd.set_option('display.float_format', lambda x: f'{x:.12e}')

In [ ]:
def generate_Table(Tabelle:pd.DataFrame, captionIN:str, labelIN:str):
    # Probleme: \cdot 10^-1; , statt . ; Nachkommastellen chekcken, Title unten?
    latex = Tabelle.to_latex(
        index=True,
        bold_rows=True,
        escape=False,
        float_format="{:.2e}".format,
        decimal=',',
        column_format=("l"*(len(Tabelle.columns.to_list()))), # L eft, C enter, R ight
        header=Tabelle.columns.to_list(),
        caption=captionIN,
        label="tab:"+labelIN,
        position="H"
    )

    return (latex)

In [ ]:
r = 0.25e-3/2 # Radius in m
A = np.pi * r**2 # Querschnittsfläche in m^2

# Laden und Ordnen der Daten
messdaten = pd.read_excel("V44.xlsx", sheet_name="Teil 2")
messdaten = messdaten.drop(columns=["Schaltung 3", "Mesung"])#, "Bereich", "Wiederstand", "spez R"])  # Entferne die Spalten
messdaten.sort_values(by="Länge (m)", inplace=True)  # Sortiere die Daten nach Länge

# Fügt die Spalten für Widerstand und spezifischen Widerstand hinzu
messdaten[r"Wiederstand ($\Omega$)"] = messdaten["U(V)"] / messdaten["I(A)"]  # Berechne den Widerstand
messdaten[r"spez R"] = messdaten[r"Wiederstand ($\Omega$)"] * (A / messdaten["Länge (m)"])  # Berechne den spezifischen Widerstand

messdaten.plot(x=r"Länge (m)", y=r"Wiederstand ($\Omega$)", kind="scatter", legend=True, grid=True)
# print(messdaten)

In [39]:
dL = 0.8e-3 # Konstanter Fehler der Länge
dA = 2e-9

# Fehler Berechnung
messdaten[r"Fehler U"] = 9e-4 + (0.005 * messdaten["U(V)"]) + 0.001

messdaten[r"Fehler I"] = 0
# Unterschiedliche Auflösungen
for i in range(len(messdaten)):
    if messdaten["Bereich"].iloc[i] == "250mA":
        messdaten[r"Fehler I"].iloc[i] = 0.002 + (0.02 * messdaten["I(A)"].iloc[i])
    elif messdaten["Bereich"].iloc[i] == "1A":
        messdaten[r"Fehler I"].iloc[i] = 0.008 + (0.02 * messdaten["I(A)"].iloc[i])
    elif messdaten["Bereich"].iloc[i] == "10A":
        messdaten[r"Fehler I"].iloc[i] = 0.08 + (0.02 * messdaten["I(A)"].iloc[i])


# Fehler Wiederstand 
messdaten[r"Fehler R"] = np.sqrt((messdaten[r"Fehler U"]/messdaten["I(A)"])**2 + ((messdaten["U(V)"]*messdaten[r"Fehler I"])/(messdaten["I(A)"]**2))**2)  

# Fehler spezifischer Widerstand
messdaten[r"Fehler spez R"] = np.sqrt(((A*messdaten[r"Fehler R"])/messdaten["Länge (m)"])**2 +
                                    ((messdaten[r"Wiederstand ($\Omega$)"] * A * dL) / (messdaten["Länge (m)"]**2))**2+
                                    ((messdaten[r"Wiederstand ($\Omega$)"] * dA) / (messdaten["Länge (m)"]))**2)


# Gewichteten Mittelwert spezifischer Wiederstand berechnen 
w = 1/(messdaten[r"Fehler spez R"].to_numpy())**2
rho = messdaten[r"spez R"].to_numpy()
rho_bar_w = (np.sum(w*rho)) / (np.sum(w))

# Fehler
del_rho_bar_w = 1 / np.sqrt(np.sum(w))

print("Gewichteter Mittelwert Rho: ", rho_bar_w)
print("Fehler Gewichteter Mittelwert Rho: ", del_rho_bar_w)
print(messdaten)


cols = ['Länge (m)', 'U(V)', 'Fehler U', 'I(A)', 'Fehler I', r'Wiederstand ($\Omega$)', 'Fehler R', 'spez R', 'Fehler spez R']
messdaten_tex = messdaten.drop(columns=['Wiederstand'])[cols]
print(generate_Table(messdaten_tex, "Messreihe 2 I const. variable Länge", "LngeVariabel"))

Gewichteter Mittelwert Rho:  1.930773546781262e-08
Fehler Gewichteter Mittelwert Rho:  6.621313015322778e-10
            Länge (m)               U(V)  I(A) Bereich        Wiederstand  \
1  5.000000000000e-02 2.000000000000e-02     1     10A 2.000000000000e-02   
2  1.000000000000e-01 3.800000000000e-02     1     10A 3.800000000000e-02   
3  1.500000000000e-01 5.800000000000e-02     1     10A 5.800000000000e-02   
4  2.000000000000e-01 7.800000000000e-02     1     10A 7.800000000000e-02   
5  2.500000000000e-01 1.000000000000e-01     1     10A 1.000000000000e-01   
0  2.560000000000e-01 1.020000000000e-01     1     10A 1.020000000000e-01   
6  3.000000000000e-01 1.180000000000e-01     1     10A 1.180000000000e-01   
7  3.500000000000e-01 1.400000000000e-01     1     10A 1.400000000000e-01   
8  4.000000000000e-01 1.570000000000e-01     1     10A 1.570000000000e-01   
9  4.500000000000e-01 1.780000000000e-01     1     10A 1.780000000000e-01   
10 5.000000000000e-01 1.960000000000e-01    

C:\Users\oskar\AppData\Local\Temp\ipykernel_14772\3103257056.py:15: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  messdaten[r"Fehler I"].iloc[i] = 0.08 + (0.02 * messdaten["I(A)"].iloc[i])
C:\Users\oskar\AppData\Local\Temp\ipykernel_14772\31

AttributeError: 'int' object has no attribute 'replace'

In [ ]:
l = messdaten[r"Länge (m)"].to_numpy()
R = messdaten[r"Wiederstand ($\Omega$)"].to_numpy()
error = messdaten[r"Fehler R"] # Fehler in Ohm

# simpy Linear Fit 
linfit = scipy.stats.linregress(l, R)

Residuendiagramm_manuell(l, R, error, linfit.slope, linfit.intercept, linfit.stderr, linfit.intercept_stderr, x_label="Länge (m)", y_label=r"Widerstand ($\Omega$)")
print(f"end=> Steigung: {linfit.slope:.4f} ± {linfit.stderr:.4f}, Achsenabschnitt: {linfit.intercept:.4f} ± {linfit.intercept_stderr:.4f}")


In [ ]:
# Werte Berechnen
print("Spezifischer Widerstand: ", np.mean(messdaten[r"spez R"]))

In [ ]:
r = 0.25e-3/2 # Radius in m
A = np.pi * r**2 # Querschnittsfläche in m^2
dA = 2e-9
dl = 0.8e-3

sheets_Versuch = ["11", "12", "21", "22", "31", "32"]

for sheet in sheets_Versuch:
    Tabelle = pd.read_excel("V44.xlsx", sheet_name=sheet)
    Tabelle.sort_values(by="U(V)", inplace=True)  # Sortiere die Daten nach Länge
    
    # Fügt die Spalten für Widerstand und spezifischen Widerstand hinzu
    Tabelle[r"Wiederstand ($\Omega$)"] = Tabelle["U(V)"] / Tabelle["I(A)"]  # Berechne den Widerstand
    Tabelle[r"spez R"] = Tabelle[r"Wiederstand ($\Omega$)"] * (A / Tabelle["Länge"])  # Berechne den spezifischen Widerstand

    # Fehler Berechnung
    if sheet in ["11", "21", "31"]: # Volt digital, Ampere analog
        Tabelle[r"Fehler U"] = 9e-4 + (0.005 * Tabelle["U(V)"]) + 0.001

        Tabelle[r"Fehler I"] = 0

        # Unterschiedliche Auflösungen
        for i in range(len(Tabelle)):
            if Tabelle["Bereich"].iloc[i] == "250mA":
                Tabelle[r"Fehler I"].iloc[i] = 0.002 + (0.02 * Tabelle["I(A)"].iloc[i])
            elif Tabelle["Bereich"].iloc[i] == "1A":
                Tabelle[r"Fehler I"].iloc[i] = 0.008 + (0.02 * Tabelle["I(A)"].iloc[i])
            elif Tabelle["Bereich"].iloc[i] == "10A":
                Tabelle[r"Fehler I"].iloc[i] = 0.08 + (0.02 * Tabelle["I(A)"].iloc[i])

    else: # Ampere digital, Volt analog
        Tabelle[r"Fehler U"] = 0.02 + (0.015 * Tabelle["U(V)"])
        Tabelle[r"Fehler I"] = 9e-4 + (0.015 * Tabelle["I(A)"]) + 0.001

    # Fehler Wiederstand 
    Tabelle[r"Fehler R"] = np.sqrt((Tabelle[r"Fehler U"]/Tabelle["I(A)"])**2 + ((Tabelle["U(V)"]*Tabelle[r"Fehler I"])/(Tabelle["I(A)"]**2))**2)  

    # Fehler spezifischer Widerstand
    Tabelle[r"Fehler spez R"] = np.sqrt(((A*Tabelle[r"Fehler R"])/Tabelle["Länge"])**2 +
                                        ((Tabelle[r"Wiederstand ($\Omega$)"] * A * dl) / (Tabelle["Länge"]**2))**2+
                                        ((Tabelle[r"Wiederstand ($\Omega$)"] * dA) / (Tabelle["Länge"]))**2)

    # print(Tabelle)


    # Gewichteten Mittelwert spezifischer Wiederstand berechnen 
    w = 1/(Tabelle[r"Fehler spez R"].to_numpy())**2
    rho = Tabelle[r"spez R"].to_numpy()
    rho_bar_w = (np.sum(w*rho)) / (np.sum(w))

    # Fehler
    del_rho_bar_w = 1 / np.sqrt(np.sum(w))

    # print("Gewichteter Mittelwert Rho: ", rho_bar_w, " Tabelle ", sheet)
    # print("Fehler Gewichteter Mittelwert Rho: ", del_rho_bar_w, " Tabelle ", sheet)

    # Print Latex Tabellen
    cols = ['U(V)', 'Fehler U', 'I(A)', 'Fehler I', r'Wiederstand ($\Omega$)', 'Fehler R', 'spez R', 'Fehler spez R']
    Tabelle = Tabelle.drop(columns=["Bereich", "Länge"])[cols]
    string_Tabelle = (generate_Table(Tabelle, sheet, "Messreihe"+sheet))
    # string_Tabelle.replace("e", r"$\cdot 10^{")
    # string_Tabelle.replace("&", r"}$ &")

    print(string_Tabelle)
    print("")
    print(r"Gewichteter Mittelwert " + str(rho_bar_w) + r" $\bar{\rho}=Eins\Omega \,\text{m}$ und gewichteter mittlerer Fehler " + str(del_rho_bar_w) + r" $\Delta \bar{\rho}=Zwei\Omega \,\text{m}$.")
    print()
        
    
    # Tabelle.plot(x=r"Länge", y=r"Wiederstand ($\Omega$)", kind="scatter", legend=True, grid=True)